In [ ]:
import os
import duckdb

train_path = r"C:\Users\samer\research\Agentic AI on edge AI\Towards cloud independent agentic edge\data\train.parquet"

# DuckDB won't create missing parent folders for its spill directory
os.makedirs(r"C:\tmp\duck", exist_ok=True)

# One connection with a memory cap; duckdb spills to disk instead of crashing
con = duckdb.connect()
con.execute("SET memory_limit='4GB'")
con.execute("SET temp_directory='C:/tmp/duck'")
# Order-preserving export buffers rows in RAM; turning it off lets COPY truly stream
con.execute("SET preserve_insertion_order=false")
con.execute("SET threads=4")  # fewer threads = fewer parallel buffers

print("=== STEP 2: EXPLORE VIA DUCKDB ===")

# 1. Basic summary using 'id' and 'date'
print(con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS total_subjects,
        MIN(date) AS start_time,
        MAX(date) AS end_time
    FROM read_parquet('{train_path}')
""").df())

# 2. Breakdown of insulin delivery modalities
print("\n--- Modality Breakdown ---")
print(con.sql(f"""
    SELECT insulin_delivery_modality, COUNT(DISTINCT id) AS unique_patients, COUNT(*) AS row_count
    FROM read_parquet('{train_path}')
    GROUP BY insulin_delivery_modality
""").df())

print("\n=== STEP 3: FILTER (Pumps Only) ===")

# Lazy view: nothing is loaded into RAM until you query it.
# Filters out MDI (Multiple Daily Injections) to keep pump users only.
con.execute(f"""
    CREATE OR REPLACE VIEW pumps AS
    SELECT *
    FROM read_parquet('{train_path}')
    WHERE insulin_delivery_modality != 'MDI' OR insulin_delivery_modality IS NULL
""")

print(con.sql("""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT id) AS remaining_subjects FROM pumps
""").df())

# When you need pandas, select only needed columns and/or sample, e.g.:
# df = con.sql("SELECT id, date, insulin_delivery_modality FROM pumps USING SAMPLE 10%").df()


In [ ]:
# Export the filtered (pumps-only) data to Parquet (a CSV would be 20+ GB and fill the disk).
# Needs `con` and the `pumps` view from the cell above.
# COPY streams straight to disk, so the data is never loaded fully into RAM.
import os
out_path = r"C:\Users\samer\research\Agentic AI on edge AI\Towards cloud independent agentic edge\data\pumps_only.parquet"

con.execute(f"COPY (SELECT * FROM pumps) TO '{out_path}' (FORMAT PARQUET, COMPRESSION ZSTD)")

print(f"Saved: {out_path}")
print(f"Size: {os.path.getsize(out_path) / 1024**3:.2f} GB")
print(con.sql(f"SELECT COUNT(*) AS rows_written FROM read_parquet('{out_path}')").df())


In [ ]:
import duckdb

train_path = r"C:\Users\samer\research\Agentic AI on edge AI\Towards cloud independent agentic edge\data\train.parquet"

# 1. Print all exact column names and types
cols_df = duckdb.query(f"DESCRIBE SELECT * FROM read_parquet('{train_path}')").to_df()
print("=== EXACT COLUMNS IN TRAIN.PARQUET ===")
for idx, row in cols_df.iterrows():
    print(f"{row['column_name']}: {row['column_type']}")

# 2. Preview 3 rows to see actual values
print("\n=== SAMPLE ROWS ===")
sample_df = duckdb.query(f"SELECT * FROM read_parquet('{train_path}') LIMIT 3").to_df()
print(sample_df.T)  # Transposed so every column is easy to read
